<div style='text-align: center; padding: 28px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 16px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.25);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.3em; font-weight: 700;'>🎙️ Cohere Transcribe - High-Speed ASR</h1>
  <h3 style='color: #f3f4f6; margin: 0 0 6px 0; font-weight: 400;'>Colab & Kaggle T4 Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #e5e7eb; margin: 0; font-size: 0.95em;'>Ungated SOTA 2B Conformer Model · 14 Languages · Long-form Audio · Native FP16 with SDPA</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br><br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy" target="_blank">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site" target="_blank">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logo=white" />
  </a>
</div>

---

### ⚡ Quickstart Instructions
- **Google Colab**: Runtime -> Change runtime type -> **T4 GPU** -> Run all cells.
- **Kaggle**: Settings -> Accelerator -> **GPU T4 x2** -> Run all cells.
- **Ungated Model**: No Hugging Face token or approval form required! Ready to run immediately.

In [ ]:
#@title 📦 [1] Setup Environment & Install Dependencies
#@markdown Installs core speech recognition libraries and optimizes PyTorch for NVIDIA T4 GPU.

import sys
import subprocess

print("⏳ Installing required dependencies (transformers, soundfile, librosa, gradio)...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.49.0", "torch", "soundfile", "librosa",
        "sentencepiece", "protobuf", "gradio"
    ],
    check=True
)

import torch

print("\n" + "=" * 58)
print("🚀 AIQUEST Academy - Environment Configuration")
print("=" * 58)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    # Optimize PyTorch CUDA kernels for NVIDIA Turing T4 architecture
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print("✅ PyTorch SDPA & FP16 hardware acceleration enabled!")
else:
    print("⚠️ No GPU detected. Hardware accelerator recommended:")
    print("   • Colab: Runtime -> Change runtime type -> T4 GPU")
    print("   • Kaggle: Settings -> Accelerator -> GPU T4 x2")
print("=" * 58 + "\n")

In [ ]:
#@title 🎙️ [2] Load Model & Launch Gradio WebUI
#@markdown Loads the ungated Cohere Transcribe model and launches the interactive transcription studio.

import os
import time
import traceback
import torch
import librosa
import soundfile as sf
import numpy as np
import gradio as gr

# ── 1. Model Configuration ───────────────────────────────────────────────────
MODEL_ID = "evewashere/cohere-transcribe-03-2026-ungated"

print(f"⏳ Loading ungated Cohere Transcribe from '{MODEL_ID}'...")

# Use official transformers implementation (trust_remote_code=False) for exact architecture compatibility
try:
    from transformers.models.cohere_asr import CohereAsrProcessor, CohereAsrForConditionalGeneration
    processor = CohereAsrProcessor.from_pretrained(MODEL_ID)
    model = CohereAsrForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        attn_implementation="sdpa",
    )
except Exception:
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=False)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=False,
        attn_implementation="sdpa",
    )

model.eval()
print("✅ Cohere Transcribe (2B Conformer) ready in native FP16 with SDPA!")

# ── 2. Supported Languages ───────────────────────────────────────────────────
LANGUAGES = [
    ("🇺🇸 English", "en"),
    ("🇫🇷 French", "fr"),
    ("🇩🇪 German", "de"),
    ("🇪🇸 Spanish", "es"),
    ("🇵🇹 Portuguese", "pt"),
    ("🇮🇹 Italian", "it"),
    ("🇳🇱 Dutch", "nl"),
    ("🇵🇱 Polish", "pl"),
    ("🇬🇷 Greek", "el"),
    ("🇸🇦 Arabic", "ar"),
    ("🇯🇵 Japanese", "ja"),
    ("🇰🇷 Korean", "ko"),
    ("🇨🇳 Chinese (Mandarin)", "zh"),
    ("🇻🇳 Vietnamese", "vi"),
]
LANGUAGE_MAP = {code: label for label, code in LANGUAGES}

# ── 3. High-Speed Transcription Pipeline ─────────────────────────────────────
def transcribe(audio_path, language, punctuation, use_compile):
    if not audio_path:
        print("[WARN] Transcribe called without audio input.", flush=True)
        return (
            "⚠️ **No Audio Detected**: Please upload an audio file or record from microphone.",
            ""
        )

    print("\n" + "=" * 65, flush=True)
    print("🎙️ [REQUEST] Received audio for transcription", flush=True)
    print(f"📁 Audio Path: {audio_path}", flush=True)
    print(f"🌐 Target Language: {language} | 🔤 Punctuation: {punctuation} | ⚡ Compile: {use_compile}", flush=True)

    try:
        # 1. Load and resample audio
        print("⏳ [1/4] Loading audio and resampling to 16kHz mono...", flush=True)
        try:
            info = sf.info(audio_path)
            duration = float(info.duration)
        except Exception:
            duration = 0.0

        audio_data, sr = librosa.load(audio_path, sr=16000, mono=True)
        duration = float(len(audio_data)) / 16000.0
        print(f"   Audio Duration: {duration:.2f} seconds ({len(audio_data)} samples)", flush=True)

        # 2. Extract features and process chunking
        print(f"⏳ [2/4] Feature extraction with CohereAsrProcessor (language='{language}')...", flush=True)
        inputs = processor(
            audio_data,
            sampling_rate=16000,
            return_tensors="pt",
            language=language,
            punctuation=punctuation,
        )

        # Extract chunk indices if present
        audio_chunk_index = inputs.pop("audio_chunk_index", None)

        # Filter strictly for valid model kwargs accepted by model.generate (drops 'length')
        valid_kwargs = {"input_features", "attention_mask", "decoder_input_ids", "decoder_attention_mask"}
        model_inputs = {k: v for k, v in inputs.items() if k in valid_kwargs}

        # Align input_features dimension: Parakeet Conformer expects (batch, time, 128)
        if "input_features" in model_inputs:
            feat = model_inputs["input_features"]
            if feat.ndim == 3 and feat.shape[1] == 128 and feat.shape[2] != 128:
                print(f"   [INFO] Aligning feature dimensions: {tuple(feat.shape)} -> (batch, time, 128)", flush=True)
                model_inputs["input_features"] = feat.transpose(1, 2)

        # Build decoder prompt tokens if not already provided by processor
        if "decoder_input_ids" not in model_inputs:
            if hasattr(processor, "get_decoder_prompt_ids"):
                prompt_ids = processor.get_decoder_prompt_ids(language=language, punctuation=punctuation)
            else:
                pnc_token = "<|pnc|>" if punctuation else "<|nopnc|>"
                prompt_tokens = [
                    "▁",
                    "<|startofcontext|>",
                    "<|startoftranscript|>",
                    "<|emo:undefined|>",
                    f"<|{language}|>",
                    f"<|{language}|>",
                    pnc_token,
                    "<|noitn|>",
                    "<|notimestamp|>",
                    "<|nodiarize|>",
                ]
                prompt_ids = processor.tokenizer.convert_tokens_to_ids(prompt_tokens)

            batch_size = model_inputs["input_features"].shape[0]
            model_inputs["decoder_input_ids"] = torch.tensor(
                [prompt_ids] * batch_size,
                dtype=torch.long,
                device=model.device,
            )

        # Move tensors to model device and appropriate dtype
        for k, v in model_inputs.items():
            if isinstance(v, torch.Tensor):
                if torch.is_floating_point(v):
                    model_inputs[k] = v.to(device=model.device, dtype=model.dtype)
                else:
                    model_inputs[k] = v.to(device=model.device)

        # Optional torch.compile acceleration
        if use_compile and hasattr(model, "_setup_compile"):
            model._setup_compile(processor=processor)

        # 3. Model generation
        print(f"⏳ [3/4] Running inference on {model.device} with SDPA attention...", flush=True)
        start_time = time.time()
        with torch.inference_mode():
            outputs = model.generate(**model_inputs, max_new_tokens=512)
        elapsed = time.time() - start_time

        # 4. Decode outputs and reassemble text
        print("⏳ [4/4] Decoding generated tokens into text...", flush=True)
        try:
            if audio_chunk_index is not None:
                text = processor.decode(
                    outputs,
                    skip_special_tokens=True,
                    audio_chunk_index=audio_chunk_index,
                    language=language,
                )
            else:
                text = processor.decode(outputs, skip_special_tokens=True)
        except TypeError:
            text = processor.batch_decode(outputs, skip_special_tokens=True)

        if isinstance(text, (list, tuple)):
            transcription_text = text[0] if len(text) == 1 else " ".join(text)
        else:
            transcription_text = str(text)

        transcription_text = transcription_text.strip()

        # Telemetry metrics
        rtfx = duration / elapsed if elapsed > 0 else 0.0
        vram_info = ""
        vram_log = ""
        if torch.cuda.is_available():
            vram_gb = torch.cuda.memory_allocated() / (1024**3)
            vram_info = f" | 💾 **VRAM**: `{vram_gb:.2f} GB`"
            vram_log = f" | VRAM: {vram_gb:.2f} GB"

        lang_display = LANGUAGE_MAP.get(language, language)
        pnc_status = "✅ Enabled" if punctuation else "❌ Disabled"
        compile_status = "⚡ On" if use_compile else "Off"

        print(f"✅ [DONE] Finished in {elapsed:.2f}s | Real-Time Factor (RTFx): {rtfx:.1f}x{vram_log}", flush=True)
        print(f"📝 Transcript Output:\n>>> {transcription_text}\n", flush=True)
        print("=" * 65 + "\n", flush=True)

        stats_card = (
            f"⚡ **RTFx**: `{rtfx:.1f}x real-time` | "
            f"⏱️ **Latency**: `{elapsed:.2f}s` | "
            f"🎵 **Duration**: `{duration:.1f}s` | "
            f"🌐 **Language**: `{lang_display}` | "
            f"🔤 **Punctuation**: `{pnc_status}` | "
            f"⚙️ **Compilation**: `{compile_status}`"
            f"{vram_info}"
        )

        return stats_card, transcription_text

    except Exception as e:
        print(f"\n❌ [ERROR] Transcription failed: {str(e)}", flush=True)
        traceback.print_exc()
        print("=" * 65 + "\n", flush=True)
        return f"❌ **Error during transcription**: {str(e)}", ""

# ── 4. AIQUEST Academy Glassmorphic Styling ───────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
* { font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; }
.gradio-container { max-width: 1050px !important; margin: auto !important; }
.brand-header {
    text-align: center;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 28px 20px;
    border-radius: 16px;
    margin-bottom: 22px;
    box-shadow: 0 10px 28px rgba(102, 126, 234, 0.28);
}
.brand-title { color: #ffffff; font-size: 2.1em; font-weight: 700; margin: 0 0 6px 0; letter-spacing: -0.5px; }
.brand-subtitle { color: rgba(255, 255, 255, 0.92); font-size: 1.05em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    min-width: 155px;
    padding: 10px 18px;
    border-radius: 10px;
    font-weight: 600;
    font-size: 13px;
    text-decoration: none !important;
    color: white !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}
.social-btn:hover { transform: translateY(-2px); }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 14px rgba(255, 0, 0, 0.35); }
.x-btn   { background: #111827; box-shadow: 0 4px 14px rgba(0, 0, 0, 0.3); border: 1px solid rgba(255, 255, 255, 0.15); }
.sup-btn { background: linear-gradient(135deg, #f59e0b, #ea580c); box-shadow: 0 4px 14px rgba(245, 158, 11, 0.35); }
.stats-box {
    background: rgba(102, 126, 234, 0.08);
    border: 1px solid rgba(102, 126, 234, 0.22);
    border-radius: 12px;
    padding: 12px 18px;
    margin-bottom: 14px;
    font-size: 0.95em;
}
button.primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
    box-shadow: 0 4px 16px rgba(102, 126, 234, 0.35) !important;
}
"""

BRAND_HEADER_HTML = f"""
<style>
{CSS}
</style>
<div class="brand-header">
  <div class="brand-title">🎙️ Cohere Transcribe - High-Speed ASR</div>
  <div class="brand-subtitle">Colab & Kaggle T4 Edition | Created by <strong>AIQUEST Academy</strong> · 14 Languages · Ungated 2B Conformer</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe on YouTube</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

BRAND_FOOTER_HTML = """
<div style="text-align: center; margin-top: 24px; padding-top: 16px; border-top: 1px solid rgba(156, 163, 175, 0.2);">
  <div class="btn-row" style="margin-bottom: 12px;">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe on YouTube</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
  <p style="color: #6b7280; font-size: 13px; margin: 4px 0 0 0;">
    ⚡ Developed with ❤️ by <strong>AIQUEST Academy</strong> · <a href="https://aiquest.site" target="_blank" style="color: #764ba2; text-decoration: none; font-weight: 600;">aiquest.site</a> · © All rights reserved
  </p>
</div>
"""

# ── 5. Gradio Interface Construction ─────────────────────────────────────────
with gr.Blocks(title="Cohere Transcribe - AIQUEST Academy") as demo:
    gr.HTML(BRAND_HEADER_HTML)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            gr.Markdown("### 🎙️ Audio Input & Controls")
            audio_input = gr.Audio(
                label="Upload or Record Speech",
                type="filepath",
                sources=["microphone", "upload"],
            )
            language = gr.Dropdown(
                label="Spoken Language",
                choices=LANGUAGES,
                value="en",
                info="Select the language spoken in the recording.",
            )
            punctuation = gr.Checkbox(
                value=True,
                label="Smart Punctuation & Capitalization",
                info="Generates properly punctuated and capitalized text.",
            )
            use_compile = gr.Checkbox(
                value=False,
                label="Enable Fast Compilation (torch.compile)",
                info="Accelerates encoder layers (one-time ~1 min warmup on first run).",
            )
            with gr.Row():
                btn = gr.Button("⚡ Transcribe Audio", variant="primary", size="lg")
                clear_btn = gr.Button("🔄 Reset", variant="secondary", size="lg")

        with gr.Column(scale=7):
            gr.Markdown("### 📝 Transcript & Telemetry")
            stats_box = gr.Markdown(
                value="*Speed metrics and telemetry will be displayed here after transcribing.*",
                elem_classes=["stats-box"],
            )
            output = gr.Textbox(
                label="Transcription Output",
                lines=15,
                max_lines=25,
                interactive=False,
                placeholder="The transcription result will appear here...",
            )
            with gr.Row():
                copy_btn = gr.Button("📋 Copy Transcript", variant="secondary", size="sm")

    def reset_ui():
        return (
            None,
            "en",
            True,
            False,
            "*Speed metrics and telemetry will be displayed here after transcribing.*",
            ""
        )

    btn.click(
        fn=transcribe,
        inputs=[audio_input, language, punctuation, use_compile],
        outputs=[stats_box, output],
    )
    clear_btn.click(
        fn=reset_ui,
        outputs=[audio_input, language, punctuation, use_compile, stats_box, output],
    )
    copy_btn.click(
        fn=None,
        inputs=[output],
        js="(text) => { if(text) { navigator.clipboard.writeText(text); } }"
    )

    gr.HTML(BRAND_FOOTER_HTML)

print("\n" + "=" * 65, flush=True)
print("🚀 Launching AIQUEST Cohere Transcribe WebUI...", flush=True)
print("🌐 Embedded preview disabled (inline=False) to stream real-time logs.", flush=True)
print("🌐 Click the public URL below to open the WebUI in a new browser tab.", flush=True)
print("=" * 65 + "\n", flush=True)

demo.queue().launch(
    inline=False,
    share=True,
    debug=True,
)